## Taiwanese Bankruptcy

Commercial loans (generally banks lending to businesses) require significant evaluation prowess given the large amounts at stake and the unique, complex attributes of each borrower. A bank's financial success can vary significantly with its ability to differentiate the risk of the requests and price the loans appropriately (via setting the interest rate) or deny the request.

Researchers exploring commercial loans collected 95 variables on almost 7000 firms in Taiwan and made this data available to the public. The abstract to their research is provided below.

>Effective bankruptcy prediction is critical for financial institutions to make appropriate lending decisions. In general, the input variables (or features), such as financial ratios, and prediction techniques, such as statistical and machine learning techniques, are the two most important factors affecting the prediction performance. While many related works have proposed novel prediction techniques, very few have analyzed the discriminatory power of the features related to bankruptcy prediction. In the literature, in addition to financial ratios (FRs), corporate governance indicators (CGIs) have been found to be another important type of input variable. However, the prediction performance obtained by combining CGIs and FRs has not been fully examined. Only some selected CGIs and FRs have been used in related studies and the chosen features may differ from study to study. Therefore, the aim of this paper is to assess the prediction performance obtained by combining seven different categories of FRs and five different categories of CGIs. The experimental results, based on a real-world dataset from Taiwan, show that the FR categories of solvency and profitability and the CGI categories of board structure and ownership structure are the most important features in bankruptcy prediction. 

>Liang, D., Lu, C.-C., Tsai, C.-F., and Shih, G.-A. (2016) _Financial Ratios and Corporate Governance Indicators in Bankruptcy Prediction: A Comprehensive Study_, European Journal of Operational Research (252:2), pp. 561-572.

We will create a neural net model to use in predicting bankruptcy.

### <font color='firebrick'>1) Notebook Prep</font>

In [3]:
import datetime
from datetime import date, timedelta 
import math
import os
import numpy as np
import pandas as pd
from pandas_datareader._utils import RemoteDataError
import pandas_datareader as web
#-------------------
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten
#-------------------
from sklearn.model_selection import train_test_split
from sklearn import preprocessing  
#-------------------
from scipy import stats
#-------------------


In [4]:
TEST_SIZE = 0.3 # percent or count
EPOCHS = 25
ST_DEV_FOR_OUTLIER = 3
# pd.set_option('display.max_rows', None)

### <font color='firebrick'>2) Retrieve and Review the Data</font>

In [5]:
taiwanese_bankruptcy_data_df = pd.read_csv('taiwanese_bankruptcy_data.csv') 

In [6]:
taiwanese_bankruptcy_data_df.head(12)

,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,1,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,...,0.716845,0.009219,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,1,0.016469
1,1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,...,0.795297,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794
2,1,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,...,0.774670,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1,0.016474
3,1,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,...,0.739555,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1,0.023982
4,1,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,...,0.795016,0.003878,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,1,0.035490
5,1,0.388680,0.415177,0.419134,0.590171,0.590251,0.998758,0.796903,0.808771,0.303116,...,0.710420,0.005278,0.622605,0.590172,0.829939,0.285087,0.026675,0.564538,1,0.019534
6,0,0.390923,0.445704,0.436158,0.619950,0.619950,0.998993,0.797012,0.808960,0.302814,...,0.736619,0.018372,0.623655,0.619949,0.829980,0.292504,0.026622,0.564200,1,0.015663
7,0,0.508361,0.570922,0.559077,0.601738,0.601717,0.999009,0.797449,0.809362,0.303545,...,0.815350,0.010005,0.623843,0.601739,0.841459,0.278607,0.027031,0.566089,1,0.034889
8,0,0.488519,0.545137,0.543284,0.603612,0.603612,0.998961,0.797414,0.809338,0.303584,...,0.803647,0.000824,0.623977,0.603613,0.840487,0.276423,0.026891,0.565592,1,0.065826
9,0,0.495686,0.550916,0.542963,0.599209,0.599209,0.999001,0.797404,0.809320,0.303483,...,0.804195,0.005798,0.623865,0.599205,0.840688,0.279388,0.027243,0.566668,1,0.030801


In [7]:
taiwanese_bankruptcy_data_df.shape

(6819, 96)

In [8]:
taiwanese_bankruptcy_data_df['Bankrupt?'].unique()

array([1, 0], dtype=int64)

In [9]:
taiwanese_bankruptcy_data_df.columns

Index(['Bankrupt?', ' ROA(C) before interest and depreciation before interest',
       ' ROA(A) before interest and % after tax',
       ' ROA(B) before interest and depreciation after tax',
       ' Operating Gross Margin', ' Realized Sales Gross Margin',
       ' Operating Profit Rate', ' Pre-tax net Interest Rate',
       ' After-tax net Interest Rate',
       ' Non-industry income and expenditure/revenue',
       ' Continuous interest rate (after tax)', ' Operating Expense Rate',
       ' Research and development expense rate', ' Cash flow rate',
       ' Interest-bearing debt interest rate', ' Tax rate (A)',
       ' Net Value Per Share (B)', ' Net Value Per Share (A)',
       ' Net Value Per Share (C)', ' Persistent EPS in the Last Four Seasons',
       ' Cash Flow Per Share', ' Revenue Per Share (Yuan ¥)',
       ' Operating Profit Per Share (Yuan ¥)',
       ' Per Share Net profit before tax (Yuan ¥)',
       ' Realized Sales Gross Profit Growth Rate',
       ' Operating Profit

In [10]:
taiwanese_bankruptcy_data_df.isnull().sum()

Bankrupt?                                                   0
 ROA(C) before interest and depreciation before interest    0
 ROA(A) before interest and % after tax                     0
 ROA(B) before interest and depreciation after tax          0
 Operating Gross Margin                                     0
                                                           ..
 Liability to Equity                                        0
 Degree of Financial Leverage (DFL)                         0
 Interest Coverage Ratio (Interest expense to EBIT)         0
 Net Income Flag                                            0
 Equity to Liability                                        0
Length: 96, dtype: int64

In [11]:
# pd.set_option('display.max_rows', None)
taiwanese_bankruptcy_data_df.dtypes

Bankrupt?                                                     int64
 ROA(C) before interest and depreciation before interest    float64
 ROA(A) before interest and % after tax                     float64
 ROA(B) before interest and depreciation after tax          float64
 Operating Gross Margin                                     float64
                                                             ...   
 Liability to Equity                                        float64
 Degree of Financial Leverage (DFL)                         float64
 Interest Coverage Ratio (Interest expense to EBIT)         float64
 Net Income Flag                                              int64
 Equity to Liability                                        float64
Length: 96, dtype: object

In [12]:
np.abs(stats.zscore(taiwanese_bankruptcy_data_df.loc[ : , taiwanese_bankruptcy_data_df.columns != 'Bankrupt?'])).head(3)

,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,Continuous interest rate (after tax),...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,2.217909,2.045798,2.400361,0.383334,0.382638,0.016456,0.023516,0.020165,0.087479,0.031281,...,2.254317,0.049491,0.084274,0.383465,0.861611,0.680171,0.060031,0.098978,NaN,0.622067
1,0.673828,0.311068,0.598450,0.135068,0.136307,0.014671,0.014799,0.015966,0.005957,0.009823,...,0.309033,0.049491,0.021388,0.135253,0.029804,0.240685,15.129816,0.364559,NaN,0.535573
2,1.303672,0.908420,1.319910,0.383759,0.388177,0.007858,0.061111,0.051183,0.142238,0.086566,...,0.820510,0.049491,0.005984,0.383698,0.249820,0.679247,0.062961,0.125015,NaN,0.621960


In [13]:
# np.abs(stats.zscore(taiwanese_bankruptcy_data_df.loc[ : , taiwanese_bankruptcy_data_df.columns != 'Bankrupt?'])) > ST_DEV_FOR_OUTLIER

In [14]:
(np.abs(stats.zscore(taiwanese_bankruptcy_data_df.loc[ : , 
                                                     taiwanese_bankruptcy_data_df.columns != 'Bankrupt?'])) > ST_DEV_FOR_OUTLIER).sum()


 ROA(C) before interest and depreciation before interest     88
 ROA(A) before interest and % after tax                     110
 ROA(B) before interest and depreciation after tax          101
 Operating Gross Margin                                      63
 Realized Sales Gross Margin                                 63
                                                           ... 
 Liability to Equity                                         28
 Degree of Financial Leverage (DFL)                          12
 Interest Coverage Ratio (Interest expense to EBIT)          36
 Net Income Flag                                              0
 Equity to Liability                                        106
Length: 95, dtype: int64

### <font color='firebrick'>3) Prepare the Data</font>

In [15]:
# convert to numpy arrays
X_df = taiwanese_bankruptcy_data_df.drop(['Bankrupt?'],axis=1)
X = X_df.values
y_df = taiwanese_bankruptcy_data_df['Bankrupt?']
y = y_df.values

In [16]:
type(X)

numpy.ndarray

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size = TEST_SIZE,
                                                    random_state=42)

# To create a validate data set, we could call `train_test_split` again and pass it X_train and y_train. 

In [18]:
scaler = preprocessing.MinMaxScaler() 
X_train_scaled = scaler.fit_transform(X_train)

In [19]:
y

array([1, 1, 1, ..., 0, 0, 0], dtype=int64)

### <font color='firebrick'>4) Define and Compile the Model</font>

In [20]:
# define model
model = Sequential()

model.add(Flatten(input_shape=([X_train.shape[1]]),
                  name='inputlayer'))

#from tensorflow.keras import layers

model.add(Dense(60, activation='relu', 
                kernel_initializer='he_normal', 
                name='hiddenlayer1'))

model.add(Dense(2, activation='softmax',name='outputlayer'))
# model.add(Dense(1, activation='sigmoid',name='thirdlayer')) # for use with binary_crossentropy

In [21]:
# compile the model
model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy',  
#             loss='binary_crossentropy',  # for use with sigmoid output layer
              metrics=['accuracy', 'SparseCategoricalAccuracy']
              ) 


### <font color='firebrick'>5) Fit and Evaluate the Model</font>

In [22]:
# fit the model
model.fit(X_train_scaled, 
          y_train, 
          epochs=EPOCHS,
#         validation_data=(x_val, y_val)
          batch_size=32, 
          verbose=1)


Epoch 1/25




150/150 [==============================] - 1s 1ms/step - loss: 0.1844 - accuracy: 0.9357 - sparse_categorical_accuracy: 0.9357
Epoch 2/25
150/150 [==============================] - 0s 1ms/step - loss: 0.1265 - accuracy: 0.9702 - sparse_categorical_accuracy: 0.9702
Epoch 3/25
150/150 [==============================] - 0s 1ms/step - loss: 0.1211 - accuracy: 0.9702 - sparse_categorical_accuracy: 0.9702
Epoch 4/25
150/150 [==============================] - 0s 1ms/step - loss: 0.1130 - accuracy: 0.9702 - sparse_categorical_accuracy: 0.9702
Epoch 5/25
150/150 [==============================] - 0s 1ms/step - loss: 0.1090 - accuracy: 0.9705 - sparse_categorical_accuracy: 0.9705
Epoch 6/25
150/150 [==============================] - 0s 1ms/step - loss: 0.1035 - accuracy: 0.9707 - sparse_categorical_accuracy: 0.9707
Epoch 7/25
150/150 [==============================] - 0s 1ms/step - loss: 0.0990 - accuracy: 0.9705 - sparse_categorical_accuracy: 0.9705
Epoch 8/25
150/150 [=========================

In [23]:
X_test_scaled = scaler.transform(X_test)

In [24]:
# evaluate the model
model.evaluate(X_test_scaled, y_test, verbose=0)

[0.11948925256729126, 0.9643206000328064, 0.9643206000328064]

### <font color='firebrick'>6) Deploy, Predict and Act</font>

Next steps would be to set up a solution to allow the model (specifically, the `predict` method) to be called remotely via an API. 

An external system (e.g., a loan application processing system) would now include a call to a server (probably via the REST-consistent protocol like HTTP) with the name of the function requested (e.g., `predict_payoff_success()`), passing the data, and receiving back a probability of payoff for each loan. 

In [25]:
def predict_payoff_success(dataset):
    '''Mock function that resembles what might be called remotely across systems'''
    dataset_np = dataset.values  # may not be necessary
    dataset_np_scaled = scaler.transform(dataset_np)
    prediction_array = model.predict(dataset_np_scaled) # see also model.predict_classes(X_test_scaled)
    return prediction_array[:,0] # return the probability of success (first column) or bankruptcy (second column)

#### Set Up

Our loan application system will call our model's `predict_payoff_success()` and then, based on the probability of payoff, will suggest:

* Rejecting the loan
* Approving the loan but at a higher interest rate than market (see the `risk_adjusted_rate()` function
* Approving the loan at the market rate

The market rate will be determined by the moving average of the last week of 90 day AA commercial paper rates as provided by the Federal Reserve [90-Day AA Financial Commercial Paper Interest Rate (DCPF3M)](https://fred.stlouisfed.org/series/DCPF3M , "90-Day AA Financial Commercial Paper Interest Rate (DCPF3M)").

In [26]:
today = datetime.date.today()
last_week = datetime.date.today() - datetime.timedelta(7)
commerical_paper_rate = 0

try:
    commerical_paper_rate = web.DataReader('DCPF3M','fred',last_week, today)["DCPF3M"].mean()
#    commerical_paper_rate = pdr.DataReader("DCPF3M", "fred", last_week, today)["DCPF3M"].mean()
#    commerical_paper_rate = pdr.fred.FredReader("DCPF3M", "fred", last_week, today)["DCPF3M"].mean()
except RemoteDataError:
    commerical_paper_rate = 5.21
    print('FRED access error')

if commerical_paper_rate == 0 | math.isnan(commerical_paper_rate):
    commerical_paper_rate = 5.23

print(commerical_paper_rate)

5.260000000000001


In [27]:
def risk_adjusted_rate(risk, rate):
    '''Adjusts interests rates to charge based on risk profile'''
    return rate + (1-risk)

In [28]:
new_data = pd.read_csv('taiwanese_new_data.csv')
y_pred = predict_payoff_success(new_data)
print(y_pred)

1/1 [==============================] - 0s 79ms/step
[0.9961055  0.90248746 0.97695804 0.8334518  0.9635132  0.99810106
 0.98288697 0.9974917  0.9973863  0.97666115 0.995764   0.998858
 0.98581034 0.95898354 0.9761881  0.9991986  0.99618846 0.999297
 0.96716064 0.9913743 ]


In [29]:
for _ in y_pred:
    print(f"{_:.0%}")

100%
90%
98%
83%
96%
100%
98%
100%
100%
98%
100%
100%
99%
96%
98%
100%
100%
100%
97%
99%


In [30]:
for _ in range(y_pred.shape[0]):
    if y_pred[_] < 0.40:
        print(f"Reject this loan application")
    elif y_pred[_] > 0.90:
        print(f"Approve this loan application at  {commerical_paper_rate/100:.2%}")
    else: 
        print(f"Approve this risky loan application at {risk_adjusted_rate(y_pred[_], commerical_paper_rate)/100:.4%}")
    print('―' * 50) 

Approve this loan application at  5.26%
――――――――――――――――――――――――――――――――――――――――――――――――――
Approve this loan application at  5.26%
――――――――――――――――――――――――――――――――――――――――――――――――――
Approve this loan application at  5.26%
――――――――――――――――――――――――――――――――――――――――――――――――――
Approve this risky loan application at 5.4265%
――――――――――――――――――――――――――――――――――――――――――――――――――
Approve this loan application at  5.26%
――――――――――――――――――――――――――――――――――――――――――――――――――
Approve this loan application at  5.26%
――――――――――――――――――――――――――――――――――――――――――――――――――
Approve this loan application at  5.26%
――――――――――――――――――――――――――――――――――――――――――――――――――
Approve this loan application at  5.26%
――――――――――――――――――――――――――――――――――――――――――――――――――
Approve this loan application at  5.26%
――――――――――――――――――――――――――――――――――――――――――――――――――
Approve this loan application at  5.26%
――――――――――――――――――――――――――――――――――――――――――――――――――
Approve this loan application at  5.26%
―――――――――――――――――――――――――――――――――――――――――――

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=073a5ba6-5e7f-487e-b570-b959ffadafe0' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>